In [ ]:
print(1)

In [ ]:
import os
import time
import uuid
import json
from dotenv import load_dotenv

from portkey_ai import Portkey ,createHeaders , PORTKEY_GATEWAY_URL


load_dotenv()


PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

portkey = Portkey(api_key=PORTKEY_API_KEY)


In [ ]:
GROQ_SLUG = "gkey"
GROQ_MODEL = f"@{GROQ_SLUG}/openai/gpt-oss-120b"


GROQ_SLUG_2 = "gkey"
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/openai/gpt-oss-20b"


# Keep GROQ_API_KEY for the Baseline experiment (direct Groq call — no gateway)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [ ]:
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)


### Raw LLM Calls

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Bitcoin scripts?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")


1st Portkey Experiment

In [ ]:
section("EXP 1 — Basic Gateway Call")

questions = [
    "What is AI and gen ai ?",
    "What is coffee and black coffee?",
]


for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")

print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")

In [ ]:
section("EXP 2 — Metadata & Observability")

# new unique id

session = str(uuid.uuid4())[:8]

scenarios = [
    ("pentest", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "docs-chatbot",     "How does BGP path selection work?"),
    ("sirjana", "support-bot",      "What is SRIOV virtualization?"),
    ("abscissa", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   # same user, diff Q
]


In [ ]:
for user, feature, q in scenarios:
    t0 = time.time()
    r = portkey.with_options(
        metadata={
            "_user":       user,          # powers per-user analytics in dashboard
            "session_id":  session,
            "feature":     feature,
            "environment": "notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")

In [ ]:
retry_config = {
    "retry": {
        "attempts": 3,
        "on_status_codes": [429, 500, 502, 503, 504]
    }
}

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, config=retry_config)

section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a AI?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")

In [ ]:
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms

portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config=timeout_config)

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")

print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))

In [56]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},        # primary
        {"override_params": {"model": GROQ_MODEL_SMALL}}   # fallback if primary fails
    ]
}


portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)


section("EXP 5 — Fallback Routing")
print(f"Primary  : {GROQ_MODEL}")
print(f"Fallback : {GROQ_MODEL_SMALL}\n")

fallback_questions = [
    "What is Intel Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="primary served")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")



  EXP 5 — Fallback Routing
Primary  : @gkey/openai/gpt-oss-120b
Fallback : @gkey/openai/gpt-oss-20b

❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04
❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set c

In [58]:
# ── Part 2: FORCED fallback — bad primary key triggers real failover ──
print("\n\n--- FORCED FALLBACK DEMO ---")
print("Primary target uses a deliberately invalid Groq API key.")
print("Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.\n")

forced_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "provider": "groq",
            "api_key": "gsk_FAKE_INVALID_KEY_THIS_WILL_FAIL",   # bad key → 401 from Groq
            "override_params": {"model": "llama-3.3-70b-versatile"}
        },
        {"override_params": {"model": GROQ_MODEL_SMALL}}         # real key via virtual key
    ]
}

portkey_forced = Portkey(api_key=PORTKEY_API_KEY, config=forced_fallback_config)

try:
    t0 = time.time()
    r = portkey_forced.chat.completions.create(
        messages=[{"role": "user", "content": "What is ge ai?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Got a response in {ms:.0f}ms despite the bad primary key!")
    print(f"   {r.choices[0].message.content[:250]}")
    print("\n→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED")
    print("   The fallback fired automatically — app code never saw the error.")
except Exception as e:
    print(f"❌ Both targets failed: {e}")



--- FORCED FALLBACK DEMO ---
Primary target uses a deliberately invalid Groq API key.
Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.

❌ Both targets failed: Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
